<a href="https://colab.research.google.com/github/smlk3/report-assistant/blob/main/rapor_asistani.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Rapor Asistanı — Router Mimarili RAG + Text-to-SQL

**Mimari:** PDF/Word yükle → metin & tabloları çıkar → iki yol hazırla:
1. **Yorumsal sorular** → yapı-temelli chunk + komşu genişletmeli RAG
2. **Sayısal/aggregasyon sorular** → tablolar DuckDB'ye → LLM SQL üretir → gerçek hesap

Bir **router** her soruyu doğru yola yönlendirir. Model: Qwen2.5-7B-Instruct (4-bit, T4 GPU'ya sığar).

> Çalıştırmadan önce: **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU**

## 1. Kurulum

In [1]:
# Bağımlılık çakışmalarını önlemek için starlette versiyonunu sabitleyerek kuruyoruz
!pip install -q -U transformers accelerate bitsandbytes sentence-transformers faiss-cpu \
    pymupdf python-docx pdfplumber duckdb gradio "starlette<1.0.0,>=0.49.1"
!nvidia-smi

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 87.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 102.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 86.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 139.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 93.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.

### 💾 Model Kalıcılığı (Google Drive)
Bu hücre, modelleri Google Drive'a kaydeder. Böylece runtime sıfırlansa bile 15GB veriyi tekrar indirmek zorunda kalmazsınız.

In [2]:
import os
from google.colab import drive

# 1. Drive'ı bağla
drive.mount('/content/drive')

# 2. Model kayıt klasörünü belirle
CACHE_DIR = '/content/drive/MyDrive/huggingface_cache'
os.makedirs(CACHE_DIR, exist_ok=True)

# 3. Hugging Face'e bu klasörü kullanmasını söyle
os.environ['HF_HOME'] = CACHE_DIR
os.environ['TRANSFORMERS_CACHE'] = CACHE_DIR

print(f"✅ Modeller artık şu klasöre kaydedilecek: {CACHE_DIR}")

Mounted at /content/drive
✅ Modeller artık şu klasöre kaydedilecek: /content/drive/MyDrive/huggingface_cache


## 2. LLM'i 4-bit Yükle (Qwen2.5-7B-Instruct)

In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import os

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

# 4-bit yapılandırması (Bellek tasarrufu için)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

try:
    print(f"⏳ Model yükleniyor: {MODEL_ID}...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )
    model.eval()
    print("✅ Model başarıyla yüklendi!")
except Exception as e:
    print(f"❌ Hata oluştu: {e}")
    print("İpucu: Drive bağlantınızın aktif olduğundan emin olun.")

def llm(system: str, user: str, max_new_tokens: int = 512) -> str:
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]
    # return_tensors="pt" BatchEncoding nesnesi döner, .input_ids ile tensöre erişiyoruz
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id,
        )
    # inputs[0] diyerek BatchEncoding içindeki ilk (ve tek) boyuta ulaşıyoruz
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True).strip()

# Test
try:
    print("🤖 Test cevabı:", llm("Kısa cevap ver.", "Merhaba, hazır mısın?"))
except Exception as e:
    print(f"Test sırasında hata: {e}")

⏳ Model yükleniyor: Qwen/Qwen2.5-7B-Instruct...
❌ Hata oluştu: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 
İpucu: Drive bağlantınızın aktif olduğundan emin olun.
Test sırasında hata: 


## 3. Belge Çıkarım Katmanı (PDF + Word, metin ve tablolar)

In [ ]:
import fitz  # pymupdf
import pdfplumber
from docx import Document
import pandas as pd

def pdf_oku(yol: str):
    """PDF'ten metin (PyMuPDF) ve tabloları (pdfplumber) çıkarır."""
    doc = fitz.open(yol)
    metin = "\n".join(sayfa.get_text() for sayfa in doc)
    doc.close()

    tablolar = []
    with pdfplumber.open(yol) as pdf:
        for sayfa_no, sayfa in enumerate(pdf.pages, 1):
            for t in sayfa.extract_tables():
                if t and len(t) > 1:
                    df = pd.DataFrame(t[1:], columns=t[0])
                    df.attrs["kaynak"] = f"sayfa {sayfa_no}"
                    tablolar.append(df)
    return metin, tablolar

def docx_oku(yol: str):
    d = Document(yol)
    metin = "\n".join(p.text for p in d.paragraphs if p.text.strip())
    tablolar = []
    for t in d.tables:
        satirlar = [[h.text.strip() for h in satir.cells] for satir in t.rows]
        if len(satirlar) > 1:
            tablolar.append(pd.DataFrame(satirlar[1:], columns=satirlar[0]))
    return metin, tablolar

def belge_oku(yol: str):
    if yol.lower().endswith(".pdf"):
        metin, tablolar = pdf_oku(yol)
    elif yol.lower().endswith(".docx"):
        metin, tablolar = docx_oku(yol)
    else:
        raise ValueError("Sadece .pdf ve .docx desteklenir")
    if len(metin.strip()) < 50:
        print("⚠️ Çok az metin çıktı — belge taranmış (görüntü) olabilir, OCR gerekir.")
    return metin, tablolar

## 4. Yapı-Temelli Chunking

Sabit kelime sayısı yerine: önce **başlık sınırlarından**, gereken yerde **paragraf sınırlarından** böler.
Chunk sayısı belgeye göre dinamik belirlenir; alakalı içeriğin ortadan kesilme riski azalır.

In [ ]:
import re

def yapisal_bol(metin: str, max_kelime: int = 350, min_kelime: int = 30):
    """Başlık → paragraf hiyerarşisiyle dinamik chunking."""
    # Numaralı başlıklar (1., 2.3 ...) veya tamamı büyük harf satırlar bölüm sınırı sayılır
    bolumler = re.split(
        r"\n(?=\d+(?:\.\d+)*[.)]?\s+[A-ZÇĞİÖŞÜ]|[A-ZÇĞİÖŞÜ0-9 ,;:'\-]{12,}\n)",
        metin,
    )
    chunklar = []
    for b in bolumler:
        b = b.strip()
        if not b:
            continue
        if len(b.split()) <= max_kelime:
            chunklar.append(b)
            continue
        # Bölüm büyükse paragraf sınırlarından paketle
        mevcut, sayac = [], 0
        for p in re.split(r"\n\s*\n", b):
            pk = len(p.split())
            if sayac + pk > max_kelime and mevcut:
                chunklar.append("\n\n".join(mevcut))
                mevcut, sayac = [], 0
            mevcut.append(p); sayac += pk
        if mevcut:
            chunklar.append("\n\n".join(mevcut))

    # Çok küçük kırpıntıları bir öncekiyle birleştir
    temiz = []
    for c in chunklar:
        if temiz and len(c.split()) < min_kelime:
            temiz[-1] += "\n" + c
        else:
            temiz.append(c)
    return temiz

## 5. RAG İndeksi + Komşu Genişletmeli Arama

Arama **küçük chunk** üzerinde yapılır (hassasiyet), modele chunk'ın **komşularıyla birlikte** verilir (bütünlük).

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

embedder = SentenceTransformer("BAAI/bge-m3")  # TR + EN güçlü çok dilli model

class RagIndeks:
    def __init__(self, chunklar):
        self.chunklar = chunklar
        v = embedder.encode(chunklar, normalize_embeddings=True, show_progress_bar=True)
        self.index = faiss.IndexFlatIP(v.shape[1])
        self.index.add(np.array(v, dtype="float32"))

    def ara(self, soru: str, k: int = 3, pencere: int = 1):
        sv = embedder.encode([soru], normalize_embeddings=True)
        skor, idx = self.index.search(np.array(sv, dtype="float32"), k)
        secilen = set()
        for i in idx[0]:
            secilen.update(range(max(0, i - pencere),
                                 min(len(self.chunklar), i + pencere + 1)))
        secilen = sorted(secilen)
        baglam = "\n\n".join(
            f"--- [Parça {i}] ---\n{self.chunklar[i]}" for i in secilen
        )
        return baglam, secilen, float(skor[0][0])

## 6. Tabloları DuckDB'ye Yükle (Sayısal Yol)

Sayısal sütunlar otomatik dönüştürülür; şema metni SQL üretim prompt'una girer.

In [ ]:
import duckdb

def tablolari_kaydet(tablolar):
    con = duckdb.connect()
    sema_satirlari = []
    for i, df in enumerate(tablolar):
        df = df.copy()
        # Sütun adlarını SQL-dostu yap
        df.columns = [re.sub(r"\W+", "_", str(c)).strip("_").lower() or f"kolon_{j}"
                      for j, c in enumerate(df.columns)]
        # Sayıya çevrilebilen sütunları çevir ("1.250,50" gibi TR formatı dahil)
        for c in df.columns:
            s = (df[c].astype(str)
                      .str.replace(".", "", regex=False)
                      .str.replace(",", ".", regex=False)
                      .str.replace(r"[^\d.\-]", "", regex=True))
            sayi = pd.to_numeric(s, errors="coerce")
            if sayi.notna().mean() > 0.7:
                df[c] = sayi
        ad = f"tablo_{i+1}"
        con.register(ad, df)
        kolonlar = ", ".join(f"{c} ({df[c].dtype})" for c in df.columns)
        ornek = df.head(2).to_string(index=False)
        sema_satirlari.append(f"{ad}: {kolonlar}\nÖrnek satırlar:\n{ornek}")
    return con, "\n\n".join(sema_satirlari)

## 7. Router — Soruyu Doğru Yola Yönlendir

In [ ]:
ROUTER_SYSTEM = """Görevin bir soruyu sınıflandırmak. İki seçenek var:
- HESAP: toplam, ortalama, sayma, sıralama, karşılaştırma, yüzde gibi tablo verisi üzerinde
  hesaplama gerektiren sorular
- YORUM: açıklama, özet, neden-sonuç, değerlendirme gibi metin üzerinde yorum gerektiren sorular
SADECE tek kelime yaz: HESAP veya YORUM."""

def yonlendir(soru: str, tablo_var: bool) -> str:
    if not tablo_var:
        return "YORUM"
    cevap = llm(ROUTER_SYSTEM, soru, max_new_tokens=4).upper()
    return "HESAP" if "HESAP" in cevap else "YORUM"

## 8. İki Cevap Yolu

In [ ]:
RAG_SYSTEM = """Sen bir rapor analiz asistanısın. SADECE verilen bağlama dayanarak,
Türkçe, net ve kaynak göstererek cevap ver. Bağlamda olmayan bilgiyi uydurma;
bilgi yoksa 'Raporda bu bilgiye rastlamadım' de."""

def yorum_cevabi(soru: str, rag: "RagIndeks"):
    baglam, parcalar, skor = rag.ara(soru, k=3, pencere=1)
    cevap = llm(RAG_SYSTEM, f"Bağlam:\n{baglam}\n\nSoru: {soru}", max_new_tokens=400)
    return f"{cevap}\n\n📎 Kullanılan parçalar: {parcalar}"

SQL_SYSTEM = """Sen bir DuckDB SQL uzmanısın. Verilen şemaya göre soruyu cevaplayan
TEK bir SQL sorgusu yaz. Sadece SQL yaz, açıklama ve markdown kullanma."""

def hesap_cevabi(soru: str, con, sema: str):
    sql = llm(SQL_SYSTEM, f"Şema:\n{sema}\n\nSoru: {soru}\nSQL:", max_new_tokens=200)
    sql = re.sub(r"```(sql)?|```", "", sql).strip().rstrip(";") + ";"
    try:
        sonuc = con.execute(sql).df()
    except Exception as e:
        # Hata mesajını modele geri verip bir kez düzelttir (self-correction)
        sql2 = llm(SQL_SYSTEM,
                   f"Şema:\n{sema}\n\nSoru: {soru}\nŞu SQL hata verdi:\n{sql}\nHata: {e}\nDüzeltilmiş SQL:",
                   max_new_tokens=200)
        sql = re.sub(r"```(sql)?|```", "", sql2).strip().rstrip(";") + ";"
        sonuc = con.execute(sql).df()
    aciklama = llm(
        "Sorgu sonucunu kullanıcıya Türkçe, kısa ve net bir cümleyle açıkla.",
        f"Soru: {soru}\nSQL: {sql}\nSonuç:\n{sonuc.to_string(index=False)}",
        max_new_tokens=150,
    )
    return f"{aciklama}\n\n🧮 Sonuç tablosu:\n{sonuc.to_string(index=False)}\n\n💻 SQL: {sql}"

## 9. Her Şeyi Birleştir: Asistan Sınıfı

In [ ]:
class RaporAsistani:
    def __init__(self, dosya_yolu: str):
        print(f"📄 Okunuyor: {dosya_yolu}")
        self.metin, self.tablolar = belge_oku(dosya_yolu)
        chunklar = yapisal_bol(self.metin)
        print(f"   {len(self.metin.split())} kelime → {len(chunklar)} chunk (dinamik)")
        self.rag = RagIndeks(chunklar)
        if self.tablolar:
            self.con, self.sema = tablolari_kaydet(self.tablolar)
            print(f"   {len(self.tablolar)} tablo DuckDB'ye yüklendi")
        else:
            self.con, self.sema = None, ""
            print("   Tablo bulunamadı — sadece RAG yolu aktif")

    def sor(self, soru: str) -> str:
        yol = yonlendir(soru, tablo_var=self.con is not None)
        print(f"🔀 Router kararı: {yol}")
        if yol == "HESAP":
            try:
                return hesap_cevabi(soru, self.con, self.sema)
            except Exception as e:
                print(f"   SQL yolu başarısız ({e}), RAG'e düşülüyor")
        return yorum_cevabi(soru, self.rag)

## 10. Test — Dosya Yükle ve Sor

In [ ]:
from google.colab import files
yuklenen = files.upload()  # .pdf veya .docx seçin
dosya = list(yuklenen.keys())[0]

asistan = RaporAsistani(dosya)

In [ ]:
# Yorumsal soru örneği (RAG yolu)
print(asistan.sor("Raporun ana bulguları nelerdir, kısaca özetle"))

In [ ]:
# Sayısal soru örneği (SQL yolu) — rapor tablo içeriyorsa
print(asistan.sor("Tablodaki değerlerin ortalaması nedir?"))

## 11. Gradio Arayüzü (opsiyonel)

In [ ]:
import gradio as gr

asistan_ref = {"a": None}

def yukle(dosya):
    asistan_ref["a"] = RaporAsistani(dosya.name)
    n_tablo = len(asistan_ref["a"].tablolar)
    return f"✅ Yüklendi. {n_tablo} tablo bulundu. Sorunuzu yazabilirsiniz."

def cevapla(soru, gecmis):
    if asistan_ref["a"] is None:
        return "Önce bir rapor yükleyin."
    return asistan_ref["a"].sor(soru)

with gr.Blocks(title="Rapor Asistanı") as demo:
    gr.Markdown("# 📊 Rapor Asistanı\nPDF/Word yükleyin, Türkçe veya İngilizce soru sorun.")
    dosya = gr.File(label="Rapor (.pdf / .docx)", file_types=[".pdf", ".docx"])
    durum = gr.Markdown()
    dosya.upload(yukle, dosya, durum)
    gr.ChatInterface(fn=cevapla)

demo.launch(share=True)

## Sonraki Adımlar

- **Taranmış PDF'ler:** `pytesseract` + `tesseract-ocr-tur` ile OCR ekleyin.
- **SQL yolunu güçlendirme:** Qwen2.5-Coder-7B'yi Spider veri setiyle QLoRA fine-tune edip bu notebooktaki modelin yerine koyun.
- **Yapılandırılmış çıkarım:** Belge yüklenirken sabit bir JSON özet kartı (dönem, gelir, riskler...) çıkaran üçüncü bir yol ekleyin.
- **Değerlendirme:** 20-30 soruluk bir test seti hazırlayıp router doğruluğunu ve cevap kalitesini ölçün.